In [1]:
%matplotlib inline
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
from src.utils.paths import load_paths
from src.utils.logging import setup_logger
from src.pipeline.feature_pipeline import FeaturePipeline
from src.pipeline.artifacts import default_feature_artifacts
from sklearn.metrics import roc_auc_score, precision_recall_curve, auc, precision_recall_fscore_support, confusion_matrix

paths = load_paths()
logger = setup_logger(level="INFO")

# Load Data and Predictions
logger.info("Loading data and predictions...")
# We load the ensemble predictions as they represent the final system
df_preds = pd.read_parquet(paths.artifacts_dir / "ensemble" / "preds.parquet")

# Ensure dataset column exists (merge from df_all logic if needed, but here we reconstruct)
# Load Feature Pipeline to get feature names
feature_art = default_feature_artifacts(paths.artifacts_dir / "features")
pipeline = FeaturePipeline.load(feature_art)
feature_cols = pipeline.model_feature_names()

# Load the raw features to analyze correlations
vnat_feats = pd.read_parquet(paths.data_processed / "vnat" / "features.parquet")
vnat_feats["dataset"] = "vnat"

iscx_feats = pd.read_parquet(paths.data_processed / "iscx" / "features.parquet")
iscx_feats["dataset"] = "iscx"

df_features = pd.concat([vnat_feats, iscx_feats], ignore_index=True)

# Merge predictions with features
# We merge on flow_id.
# Note: df_preds has 'dataset' if create_ensemble was run with the fix.
# If not, we get it from df_features.
cols_to_merge = feature_cols + ["flow_id"]
if "dataset" not in df_preds.columns:
    cols_to_merge.append("dataset")

df_full = pd.merge(df_preds, df_features[cols_to_merge], on="flow_id", how="left")

# Handle potential duplicate dataset columns if merge overlap
if "dataset_x" in df_full.columns:
    df_full["dataset"] = df_full["dataset_x"].fillna(df_full["dataset_y"])
    df_full = df_full.drop(columns=["dataset_x", "dataset_y"])

print(f"Full Analysis DataFrame Shape: {df_full.shape}")


2026-03-04 10:41:43 | INFO | ai-vpn-firewall | Loading data and predictions...


KeyError: "['sample_weight'] not in index"

In [ ]:
# 1. Feature Importance Analysis (using XGBoost model)
logger.info("Analyzing Feature Importance...")
model_path = paths.artifacts_dir / "xgb" / "model.json"
booster = xgb.Booster()
booster.load_model(str(model_path))

importance = booster.get_score(importance_type='gain')
imp_df = pd.DataFrame(list(importance.items()), columns=['Feature', 'Gain']).sort_values('Gain', ascending=False)

plt.figure(figsize=(10, 8))
sns.barplot(x="Gain", y="Feature", data=imp_df.head(20))
plt.title("Top 20 Features by Gain (XGBoost)")
plt.tight_layout()
plt.show()

print("Top 5 Features:")
print(imp_df.head(5))

# Check for "Dominator" features
top_gain = imp_df.iloc[0]["Gain"]
total_gain = imp_df["Gain"].sum()
print(f"\nTop feature '{imp_df.iloc[0]['Feature']}' accounts for {top_gain/total_gain:.2%} of total gain.")
if top_gain/total_gain > 0.5:
    print("WARNING: Single feature dominance detected! Potential leakage.")


In [ ]:
# 2. Cross-Dataset Performance (VNAT vs ISCX)
logger.info("Analyzing Cross-Dataset Performance on TEST set...")
test_df = df_full[df_full["split"] == "test"].copy()

# Per-Dataset Thresholds (re-implementing logic from NB10 for consistency)
val_df = df_full[df_full["split"] == "val"].copy()
val_vnat = val_df[val_df["dataset"] == "vnat"]
val_iscx = val_df[val_df["dataset"] == "iscx"]

def get_thresholds(df, label_col="label", prob_col="p_calib"):
    if df.empty: return {}
    fprs = [0.001, 0.01]
    thresholds = {}
    y_true = df[label_col].values
    y_score = df[prob_col].values
    desc_score_indices = np.argsort(y_score, kind="mergesort")[::-1]
    y_score = y_score[desc_score_indices]
    y_true = y_true[desc_score_indices]
    distinct_value_indices = np.where(np.diff(y_score))[0]
    threshold_idxs = np.r_[distinct_value_indices, y_true.size - 1]
    tps = np.cumsum(y_true)[threshold_idxs]
    fps = 1 + threshold_idxs - tps
    fpr_values = fps / fps[-1]
    for target_fpr in fprs:
        idx = np.searchsorted(fpr_values, target_fpr, side='right') - 1
        if idx < 0: idx = 0
        thr = y_score[threshold_idxs[idx]]
        thresholds[f"fpr_{target_fpr}"] = thr
    return thresholds

thr_vnat = get_thresholds(val_vnat)
thr_iscx = get_thresholds(val_iscx)

print(f"VNAT Thresholds: {thr_vnat}")
print(f"ISCX Thresholds: {thr_iscx}")

for ds, thr_dict in [("vnat", thr_vnat), ("iscx", thr_iscx)]:
    subset = test_df[test_df["dataset"] == ds]
    if len(subset) == 0: continue

    auc_val = roc_auc_score(subset["label"], subset["p_calib"])
    print(f"\nDataset: {ds.upper()} (N={len(subset)})")
    print(f"  ROC AUC: {auc_val:.4f}")
    print(f"  Mean Prob (VPN): {subset[subset['label']==1]['p_calib'].mean():.4f}")
    print(f"  Mean Prob (Non-VPN): {subset[subset['label']==0]['p_calib'].mean():.4f}")

    # Policy Metrics
    t_block = thr_dict.get("fpr_0.001", 0.99)
    t_monitor = thr_dict.get("fpr_0.01", 0.90)

    for zone, t in [("BLOCK", t_block), ("MONITOR", t_monitor)]:
        y_hat = (subset["p_calib"] >= t).astype(int)
        prec, rec, _, _ = precision_recall_fscore_support(subset["label"], y_hat, average="binary", zero_division=0)
        _, fp, _, _ = confusion_matrix(subset["label"], y_hat).ravel()
        print(f"  {zone} (p >= {t:.4f}): Recall={rec:.4f}, Precision={prec:.4f}, FP={fp}")


In [ ]:
# 3. "Easy" vs "Hard" Traffic Analysis
# We define "Hard" as samples where the model was wrong or low confidence (0.4-0.6)
logger.info("Analyzing Hard Cases...")
test_df = test_df.copy() # Avoid SettingWithCopyWarning
test_df["error"] = np.abs(test_df["label"] - test_df["p_calib"])
hard_cases = test_df[test_df["error"] > 0.5] # Wrong prediction (if thr=0.5)

print(f"\nNumber of Misclassified Test Samples: {len(hard_cases)} out of {len(test_df)}")

if len(hard_cases) > 0:
    print("\nTop Misclassified Samples:")
    cols_to_show = ["flow_id", "dataset", "label", "p_calib", "capture_id"]
    print(hard_cases[cols_to_show].head(10))

    # Check if they belong to specific captures
    print("\nMisclassifications by Capture:")
    print(hard_cases["capture_id"].value_counts().head(5))


In [ ]:
# 4. Leakage Check: Correlation with Label
logger.info("Checking for Linear Leakage...")
corrs = []
for col in feature_cols:
    if col in df_full.columns:
        c = df_full[col].corr(df_full["label"])
        corrs.append((col, c))

corr_df = pd.DataFrame(corrs, columns=["Feature", "Correlation"]).sort_values("Correlation", key=abs, ascending=False)
print("\nTop Correlations with Label:")
print(corr_df.head(10))

if corr_df.iloc[0]["Correlation"] > 0.95:
    print(f"\nWARNING: Feature '{corr_df.iloc[0]['Feature']}' has >0.95 correlation with label. Likely leakage.")
else:
    print("\nNo single feature has >0.95 linear correlation with label.")


In [ ]:
# 5. Distribution of Top Feature
top_feat = imp_df.iloc[0]["Feature"]
plt.figure(figsize=(10, 6))
sns.boxplot(x="label", y=top_feat, hue="dataset", data=df_full)
plt.title(f"Distribution of '{top_feat}' by Label and Dataset")
plt.show()
